In [ ]:
import sys
sys.path.insert(0, '../')
from neuralsd.nsd import NeuralSD
from neuralsd.wracc import WRAcc
import pandas as pd
import main
import numpy as np
import torch

In [ ]:
torch.set_default_device("cuda:0")

In [40]:
datasets = {
    "Mushrooms" : main.get_mushrooms_data(),
    "Chess" : main.get_chess_data(),
    "StudentSuccess" : main.get_student_success_data(),
    "Connect-4" : main.get_connect4_data(),
    "HealthyAging" : main.get_healthy_aging_data(),
    "CarEvaluation" : main.get_car_data(),
    "Soybean" : main.get_soybean_data(),
    "Dermatology" : main.get_dermatology_data(),
    "Depression" : main.get_depression_data(),
    "Vancomycin" : main.get_vancomycin_data(),
    "MIMIC-IV" : main.get_mimic_iv_data(),
    "Cardio" : main.get_cardio_data()
}

targets = {
    "Mushrooms": main.get_target("Mushrooms"),
    "Chess": main.get_target("Chess"),
    "StudentSuccess": main.get_target("StudentSuccess"),
    "Connect-4": main.get_target("connect4"),
    "HealthyAging": main.get_target("HealthyAging"),
    "CarEvaluation": main.get_target("Car"),
    "Soybean": main.get_target("Soybean"),
    "Dermatology": main.get_target("Dermatology"),
    "Depression": main.get_target("Depression"),
    "Vancomycin": main.get_target("Vancomycin"),
    "MIMIC-IV": main.get_target("MIMIC-IV"),
    "Cardio": main.get_target("Cardio")
}

In [ ]:
def get_quality(dataset):
    nsd = NeuralSD(WRAcc())
    nsd.fit(datasets[dataset], targets[dataset], weight_init="best", verbose=False)
    return nsd.forward()[0].max()

def get_tp_fp(dataset):
    nsd = NeuralSD(WRAcc())
    nsd.fit(datasets[dataset], targets[dataset], weight_init="best", verbose=False)
    q, tp, fp = nsd.forward(True)
    best_idx = q.argmax()
    return tp[best_idx], fp[best_idx]


In [42]:
tp_fp_nsd = {dataset: get_tp_fp(dataset) for dataset in datasets.keys()}

In [43]:
tp_fp_nsd = {d : (round(tp_fp_nsd[d][0].item()), round(tp_fp_nsd[d][1].item())) for d in tp_fp_nsd}

In [44]:
qualities = {dataset: get_quality(dataset) for dataset in datasets}

In [45]:
qualities = {dataset: qualities[dataset].item() for dataset in datasets}

In [46]:
bsd_results = {dataset : pd.read_csv("bsd_results/results_BSD_" + dataset + ".csv") for dataset in datasets}

In [47]:
max_bsd_qualities = {dataset: bsd_results[dataset]["Quality"].max() for dataset in datasets}
display(max_bsd_qualities)

{'Mushrooms': 0.1945589303753417,
 'Chess': 0.1572410992463984,
 'StudentSuccess': 0.0817155275024607,
 'Connect-4': 0.0328262436001167,
 'HealthyAging': 0.0207082833133253,
 'CarEvaluation': 0.0999228395061728,
 'Soybean': 0.0967649524132882,
 'Dermatology': 0.2081877631461077,
 'Depression': 0.1164981508230027,
 'Vancomycin': 0.1037340660330616,
 'MIMIC-IV': 0.0431725531196384,
 'Cardio': 0.0306202828571428}

In [48]:
best_bsd_result = {dataset: bsd_results[dataset].loc[bsd_results[dataset]["Quality"].idxmax()] for dataset in datasets}
tp_fp_bsd = {dataset: (best_bsd_result[dataset]["tp"], best_bsd_result[dataset]["fp"]) for dataset in datasets}
tp_fp_bsd

{'Mushrooms': (3408, 120),
 'Chess': (1195, 131),
 'StudentSuccess': (1473, 753),
 'Connect-4': (15597, 4727),
 'HealthyAging': (99, 360),
 'CarEvaluation': (576, 0),
 'Soybean': (40, 39),
 'Dermatology': (112, 5),
 'Depression': (189, 71),
 'Vancomycin': (329, 55),
 'MIMIC-IV': (7963, 5523),
 'Cardio': (6174, 1892)}

In [52]:
TP_FP = {dataset: (best_bsd_result[dataset]["TP"], best_bsd_result[dataset]["FP"]) for dataset in datasets}

In [49]:
underperformances = []
for d in qualities:
    if tp_fp_nsd[d] == tp_fp_bsd[d]:
        print(f"NSD matches BSD in {d} dataset")
    elif np.isclose(qualities[d], max_bsd_qualities[d]):
        print(f"NSD ties BSD in {d} dataset")
    elif qualities[d] >= max_bsd_qualities[d]:
        print(f" ???????? NSD outperforms BSD in {d} dataset", qualities[d] - max_bsd_qualities[d])
    else:
        print(f"BSD outperforms NSD in {d} dataset", max_bsd_qualities[d] - qualities[d])
        underperformances.append(d)

NSD matches BSD in Mushrooms dataset
NSD matches BSD in Chess dataset
BSD outperforms NSD in StudentSuccess dataset 0.00480778990084324
NSD matches BSD in Connect-4 dataset
NSD matches BSD in HealthyAging dataset
NSD matches BSD in CarEvaluation dataset
NSD matches BSD in Soybean dataset
BSD outperforms NSD in Dermatology dataset 0.004016244987064876
NSD matches BSD in Depression dataset
NSD matches BSD in Vancomycin dataset
NSD matches BSD in MIMIC-IV dataset
NSD matches BSD in Cardio dataset
